<a href="https://colab.research.google.com/github/Gidayi-dev/Dummy/blob/main/financial_inclusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_percentage_error, roc_auc_score, mean_absolute_error
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')
sample = pd.read_csv("SampleSubmission.csv")

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('\nTrain columns:', list(train.columns))

Train shape: (23524, 13)
Test shape: (10086, 12)

Train columns: ['country', 'year', 'uniqueid', 'bank_account', 'location_type', 'cellphone_access', 'household_size', 'age_of_respondent', 'gender_of_respondent', 'relationship_with_head', 'marital_status', 'education_level', 'job_type']


In [40]:
train["target"] = (train["bank_account"] == "Yes").astype(int)

pos_rate = train["target"].mean()
print(f"Positive rate (has bank account): {pos_rate:.1%}")

Positive rate (has bank account): 14.1%


In [41]:
def engineer_features(df):
  df = df.copy()
  # Age group buckets
  df["age_group"] = pd.cut(
      df["age_of_respondent"],
      bins=[0, 25, 35, 50, 65, 120],
      labels=["youth", "young_adult", "mid", "senior", "elder"]
  )

  # Household size buckets
  df["hh_size_bin"] = pd.cut(
      df["household_size"],
      bins=[0, 2, 4, 7, 100],
      labels=["small", "medium", "large", "very_large"]
  )
  # Interaction: Urban/Rural * cellphone access
  df["urban_phone"] = df["location_type"] + "_" + df["cellphone_access"]
  #Interaction: Country * education level
  df["country_edu"] = df["country"] + "_" + df["education_level"]
  # Interaction: Job type * gender
  df["job_gender"] = df["job_type"] + "_" + df["gender_of_respondent"]

  return df



In [42]:
train = engineer_features(train)
test = engineer_features(test)

FEATURES = [
    "country", "year", "location_type", "cellphone_access", "household_size", "age_of_respondent",
    "gender_of_respondent", "relationship_with_head", "marital_status", "education_level", "job_type",
    "age_group", "hh_size_bin", "urban_phone", "country_edu", "job_gender",
]

CAT_COLS = [
    "country", "location_type", "cellphone_access", "gender_of_respondent", "relationship_with_head",
    "marital_status", "education_level", "job_type", "age_group", "hh_size_bin", "urban_phone",
    "country_edu", "job_gender",
]

In [43]:
all_data = pd.concat([train[FEATURES], test[FEATURES]], axis=0, ignore_index=True)
for col in CAT_COLS:
  le = LabelEncoder()
  all_data[col] = le.fit_transform(all_data[col].astype(str))
X_train = all_data.iloc[: len(train)].copy()
X_test = all_data.iloc[len(train) :].copy()
y_train = train["target"]

print(f"\nFeature -> {X_train.shape[1]} columns")


Feature -> 16 columns


In [44]:
from sklearn import metrics
lgb_params = dict(
    objective="binary",
    metric="binary_logloss",
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=30,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    reg_alpha=0.05,
    reg_lambda=0.1,
    random_state=42,
    verbose=-1,
    n_jobs=-1,
)

In [45]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))

print("\nTraining ...")
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
  X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
  y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

  model = lgb.LGBMClassifier(**lgb_params)
  model.fit(
      X_tr, y_tr,
      eval_set=[(X_val, y_val)],
      callbacks=[
          lgb.early_stopping(80, verbose=False),
          lgb.log_evaluation(period=-1),
      ],
  )

  oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
  test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

  fold_mae = mean_absolute_percentage_error(y_val, (oof_preds[val_idx] >= 0.5).astype(int))
  print(f" Fold {fold + 1} | {model.best_iteration_} trees | MAE@0.5 = {fold_mae:.4f}")

  auc = roc_auc_score(y_train, oof_preds)
  print(f"\nOOF AUC: {auc:.4f}")


Training ...
 Fold 1 | 105 trees | MAE@0.5 = 101462605845116.4688

OOF AUC: 0.5146
 Fold 2 | 96 trees | MAE@0.5 = 99548217055585.9688

OOF AUC: 0.5572
 Fold 3 | 134 trees | MAE@0.5 = 111034549792768.9531

OOF AUC: 0.6299
 Fold 4 | 105 trees | MAE@0.5 = 78489940370750.5156

OOF AUC: 0.7306
 Fold 5 | 125 trees | MAE@0.5 = 123504326515900.1562

OOF AUC: 0.8596


In [46]:
best_t, best_mae = 0.5, 999.0
for t in np.arange(0.05, 0.95, 0.005):
  mae = mean_absolute_error(y_train, (oof_preds >= t).astype(int))
  if mae < best_mae:
    best_mae, best_t = mae, t
baseline_mae = float(y_train.mean())
improvement = (baseline_mae - best_mae) / baseline_mae * 100

print(f"Baseline MAE (all-zero): {baseline_mae:.4f}")
print(f"Best threshold: {best_t:.3f}")
print(f"Best OOf MAE: {best_mae:.4f}")
print(f"Improvement: {improvement:.1f}%")

train_id_map = {
    f"{row['uniqueid']} x {row['country']}": oof_preds[i]
    for i, row in train.iterrows()
}

test_id_map = {
    f"{row['uniqueid']} x {row['country']}": test_preds[j]
    for j, row in test.iterrows()
}

all_pred_map = {**train_id_map, **test_id_map}

sample["proba"] = sample["unique_id"].map(all_pred_map)
sample["bank_account"] = (sample["proba"] >= best_t).astype(int)

submission = sample[["unique_id", "bank_account"]].copy()
submission.to_csv("submission.csv", index=False)

print(f"\n submission.csv saved ({len(submission):,} rows)")
print(f"Predicted bank_account=1: {submission['bank_account'].sum():,}" f"({submission['bank_account'].mean():.1%})")
print("\nFirst 5 rows:")
print(submission)

Baseline MAE (all-zero): 0.1408
Best threshold: 0.460
Best OOf MAE: 0.1129
Improvement: 19.8%

 submission.csv saved (33,610 rows)
Predicted bank_account=1: 2,763(8.2%)

First 5 rows:
                    unique_id  bank_account
0          uniqueid_1 x Kenya             0
1          uniqueid_2 x Kenya             0
2          uniqueid_3 x Kenya             1
3          uniqueid_4 x Kenya             0
4          uniqueid_5 x Kenya             0
...                       ...           ...
33605  uniqueid_2998 x Uganda             0
33606  uniqueid_2999 x Uganda             0
33607  uniqueid_3000 x Uganda             0
33608  uniqueid_3001 x Uganda             0
33609  uniqueid_3002 x Uganda             0

[33610 rows x 2 columns]
